# 01 · Расчёты

Конфигурация → проверка → счёт → продолжение с чекпоинта → серия.

* Ноутбук — **один процесс**. Небольшие прогоны удобно считать прямо здесь (раздел 3).
  Большие — под MPI (раздел 4): ноутбук только запускает `mpirun … python -m cardiac_em run`
  и показывает вывод, как терминал.
* Результаты — в `runs/<имя>/` (в git не попадают): `run.json`, `series.csv`, `activation.npz`,
  снимки, XDMF для ParaView, чекпоинты.
* Дальше: `03_analysis.ipynb` — числа и таблицы, `04_visualization.ipynb` — графики и карты.

Ядро ноутбука — Python окружения `fenicsx-env` (в VS Code: *Select Kernel*).

In [ ]:
# Общая настройка: пакет cardiac_em и помощники ноутбуков доступны из любой папки проекта
import sys
from pathlib import Path

for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "notebooks" / "nbtools.py").exists():
        sys.path.insert(0, str(_p / "notebooks"))
        break
from nbtools import ROOT, RUNS, run_stream, style  # noqa: E402

print("корень проекта:", ROOT)

## 1. Конфигурация

Два способа: взять готовую из `examples/` и поправить отдельные параметры (как здесь),
или собрать `SimulationConfig` в коде. Пути параметров — те же, что у `--set` в командной
строке: `tissue_base.active.t_max`, `regions.0.overrides.cell:ATP_i`, `time.t_end_ms`, …

`QUICK = True` — маленькая постановка (3×3 мм, 100 мс, около минуты на одном ядре), чтобы
проверить, что всё работает. `QUICK = False` — пример из `examples/tnnpm_ischemia.json` целиком
(6×6 мм, 400 мс — лучше под MPI, раздел 4).

In [ ]:
from cardiac_em.config import SimulationConfig
from cardiac_em.control import apply_overrides

QUICK = True
RUN_NAME = "nb_quick" if QUICK else "nb_tnnpm_ischemia"
OUT = RUNS / RUN_NAME

cfg = SimulationConfig.from_json(ROOT / "examples" / "tnnpm_ischemia.json")

changes = {"output.out_dir": str(OUT)}
if QUICK:
    changes.update({
        # электрика 30×30 (h = 0.1 мм), механика 10×10 — вложенные сетки
        "mesh.electric.nx": 30, "mesh.electric.ny": 30,
        "mesh.electric.lx_mm": 3.0, "mesh.electric.ly_mm": 3.0,
        "mesh.mechanical.nx": 10, "mesh.mechanical.ny": 10,
        "mesh.mechanical.lx_mm": 3.0, "mesh.mechanical.ly_mm": 3.0,
        # ишемическая зона — круг в правой части
        "regions.0.params.cx": 2.2, "regions.0.params.cy": 1.5, "regions.0.params.r": 0.7,
        # один удар целиком: ПД TNNPM ~300 мс, пик силы ~150–200 мс
        # (100 мс хватает только на фронт — APD и расслабления не будет)
        "time.t_end_ms": 450.0,
        "preload.cell_relax_ms": 100.0,
        "output.snapshot_times_ms": [5.0, 15.0, 50.0, 150.0, 250.0, 300.0, 350.0, 450.0],
        "output.save_every_mech_steps": 10,
    })
cfg = apply_overrides(cfg, changes)
print(cfg.summary())

### Проверка до счёта

Предупреждения конфигурации (стимулы после конца счёта, невложенные сетки, механическая сетка
без внутренних узлов, …) и расписание шагов. DOLFINx для этого не нужен.

In [ ]:
from cardiac_em.runtime.schedule import Schedule

print(Schedule(cfg.time, cfg.output).summary())
warnings = cfg.check()
for w in warnings:
    print("  [!]", w)
if not warnings:
    print("  предупреждений нет")

## 2. Сохранить конфигурацию

JSON-файл — это полное описание прогона: его можно запустить из терминала
(`python -m cardiac_em run <файл>`), положить рядом со статьёй или изменить для серии.

In [ ]:
RUNS.mkdir(exist_ok=True)
CFG_FILE = RUNS / f"{RUN_NAME}.json"
cfg.to_json(CFG_FILE)
print("записано:", CFG_FILE.relative_to(ROOT))

## 3. Счёт в ноутбуке

`run_simulation` делает всё сразу: собирает задачу, подготавливает ткань и преднагрузку
(или продолжает с чекпоинта из `cfg.restart`), считает и пишет результаты. Первый запуск в
новом окружении дольше — DOLFINx компилирует формы (кэш сохраняется).

`OVERWRITE = True` разрешает перезаписать прежний прогон в той же папке.

In [ ]:
from cardiac_em.control import run_simulation

OVERWRITE = True
result = run_simulation(cfg, overwrite=OVERWRITE, console_every_mech_steps=10)
print("\nготово:", result.out_dir.relative_to(ROOT), f"(t = {result.t_start_ms:g} → {result.t_end_ms:g} мс)")

Объект `result.sim` — живая задача после счёта: можно посмотреть состояние, не открывая файлы.

In [ ]:
sim = result.sim
d = sim.diagnostics()
print(f"λ_f ∈ [{d['lambda_f_min']:.4f}, {d['lambda_f_max']:.4f}],  "
      f"T_act действующее до {d['t_act_actual_max']:.2f} кПа,  "
      f"потенциал ∈ [{d['u_min']:.1f}, {d['u_max']:.1f}]")

## 4. Счёт под MPI

Тот же прогон на нескольких ядрах. Ноутбук запускает команду в отдельном процессе и
показывает её вывод; ядро ноутбука при этом свободно только после завершения.
Для долгих прогонов удобнее терминал (или `nohup … &`), а за ходом счёта следить
разделом 7.

Включите `RUN_MPI = True`, чтобы запустить.

In [ ]:
from nbtools import mpirun

N_PROC = 4
RUN_MPI = False

cfg_mpi = apply_overrides(cfg, {"output.out_dir": str(RUNS / f"{RUN_NAME}_mpi")})
cfg_mpi_file = RUNS / f"{RUN_NAME}_mpi.json"
cfg_mpi.to_json(cfg_mpi_file)

cmd = [mpirun(required=RUN_MPI), "-n", N_PROC, sys.executable, "-m", "cardiac_em",
       "run", cfg_mpi_file, "--overwrite", "--every", 20]
if RUN_MPI:
    run_stream(cmd)
else:
    print("команда (RUN_MPI = False — не запускаю):\n ", " ".join(map(str, cmd)))

## 5. Продолжение с чекпоинта

Новая папка, `restart.checkpoint_path` — чекпоинт источника, `t_end` — **абсолютное** время.
Параметры можно изменить: новый протокол стимуляции, другие ишемические параметры, …
Расхождения физики с источником будут перечислены в предупреждениях.

In [ ]:
RUN_CONT = False
EXTRA_MS = 50.0

cont = apply_overrides(cfg, {
    "output.out_dir": str(RUNS / f"{RUN_NAME}_cont"),
    "restart.checkpoint_path": str(OUT / "ckpt_last.npz"),
    "time.t_end_ms": cfg.time.t_end_ms + EXTRA_MS,
})
if RUN_CONT:
    result_cont = run_simulation(cont, overwrite=True, console_every_mech_steps=10)
else:
    print("продолжение:", cont.restart.checkpoint_path, "→", cont.output.out_dir,
          f"до {cont.time.t_end_ms:g} мс (RUN_CONT = False — не запускаю)")

## 6. Параметрическая серия

Базовая конфигурация + изменяемые параметры (`grid` — все сочетания, `zip` — попарно).
Все точки проверяются **до** счёта; повторный запуск пропускает уже посчитанные.
Сначала — «сухой» прогон: список точек без счёта.

In [ ]:
from cardiac_em.control import SweepSpec, run_sweep

SWEEP_DIR = RUNS / f"{RUN_NAME}_sweep"
spec = SweepSpec(
    base=cfg, out_root=SWEEP_DIR, mode="grid",
    parameters={
        "regions.0.overrides.cell:ATP_i": [6.8, 4.5, 4.0],
        "regions.0.overrides.cell:K_o": [5.4, 9.4],
    },
)
index = run_sweep(spec, dry_run=True)

Счёт серии: в ноутбуке (последовательно, одно ядро) или под MPI через командную строку —
для этого описание серии пишется в JSON.

In [ ]:
import json

RUN_SWEEP = False            # в ноутбуке
RUN_SWEEP_MPI = False        # под MPI

if RUN_SWEEP:
    index = run_sweep(spec, console=True)

sweep_file = RUNS / f"{RUN_NAME}_sweep.json"
sweep_file.write_text(json.dumps({
    "base": cfg.to_dict(), "out_root": str(SWEEP_DIR), "mode": spec.mode,
    "parameters": spec.parameters}, ensure_ascii=False, indent=2), encoding="utf-8")
cmd = [mpirun(required=RUN_SWEEP_MPI), "-n", N_PROC, sys.executable, "-m", "cardiac_em", "sweep", sweep_file]
if RUN_SWEEP_MPI:
    run_stream(cmd)
else:
    print("под MPI:", " ".join(map(str, cmd)))

## 7. Состояние прогонов

Статус читается из `run.json` / `sweep.json` — работает и пока счёт идёт в другом процессе.

In [ ]:
import pandas as pd
from nbtools import list_runs

pd.DataFrame(list_runs())

In [ ]:
if (Path(OUT) / "run.json").exists():
    run_stream([sys.executable, "-m", "cardiac_em", "status", OUT])
else:
    print(f"{OUT} ещё не запускался")